# Setup Environment
Install `vllm` untuk serving model yang efisien, dan `pyngrok` untuk tunneling.

In [ ]:
!pip install vllm pyngrok nest-asyncio

# Hugging Face Login
Model `AITF-SR-02/ub-sr-02-qwen3.5-9b-base-5k-CPT-SFT-v2` adalah private, jadi Anda perlu login dengan Hugging Face Access Token yang memiliki akses ke repositori tersebut.

In [ ]:
from huggingface_hub import login

# Ganti dengan token Hugging Face Anda (buat di https://huggingface.co/settings/tokens)
HF_TOKEN = "hf_..."
login(token=HF_TOKEN)

# Setup Ngrok Tunnel
Gunakan ngrok untuk mengekspos port 8000 agar bisa diakses secara publik (contoh: dari Postman atau web app lokal Anda).

In [ ]:
from pyngrok import ngrok
import nest_asyncio

nest_asyncio.apply()

# Ganti dengan authtoken ngrok Anda (dapatkan dari https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = "your_ngrok_authtoken"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Buka tunnel di port 8000 (port default vLLM)
public_url = ngrok.connect(8000).public_url
print(f"\n>>> Public URL untuk API OpenAI-compatible: {public_url}/v1 <<<")
print("Gunakan URL di atas sebagai 'base_url' di client OpenAI Anda.")

# Run vLLM Server
Menjalankan server vLLM dengan model Anda. Setelah cell ini berjalan (dan model selesai diload), Anda bisa melakukan request ke URL ngrok di atas.

In [ ]:
# Menggunakan vLLM untuk serving model sebagai OpenAI-compatible API
# Parameter --max-model-len dibatasi agar sesuai dengan memori GPU Kaggle (T4 16GB)
# Jika Kaggle menggunakan 2x T4, tambahkan --tensor-parallel-size 2 jika diperlukan.

!python -m vllm.entrypoints.openai.api_server \
    --model AITF-SR-02/ub-sr-02-qwen3.5-9b-base-5k-CPT-SFT-v2 \
    --port 8000 \
    --max-model-len 4096 \
    --dtype half

# Contoh Request (Jalankan di terminal lokal atau notebook lain)
```python
from openai import OpenAI

client = OpenAI(
    base_url="<PUBLIC_URL_DARI_NGROK>/v1",
    api_key="empty" # vLLM tidak butuh api key secara default
)

response = client.chat.completions.create(
    model="AITF-SR-02/ub-sr-02-qwen3.5-9b-base-5k-CPT-SFT-v2",
    messages=[
        {"role": "user", "content": "Halo, apa kabar?"}
    ]
)

print(response.choices[0].message.content)
```